# PySpark coding challenges

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-coding-challenges.ipynb)

Practise DataFrame transformations, nulls, dates, joins, Spark SQL CTEs, windows, nested data, schema evolution, and production scenarios. Each challenge provides input data and expected output before a collapsed reference answer.

**How to practise:** run the two setup cells, then run a challenge's sample-data cell and write your solution in its practice cell. Compare with the expected output; expand **Show reference solution** only when ready. Copy the reference code into a code cell to run its assertions. All checks collect only tiny fixtures; avoid collecting production data. Results are shown in a specified order for readability, although Spark DataFrames have no inherent row order.

The setup installs PySpark in the active notebook environment if needed; a working Java runtime is also required. Each challenge creates its own data and can be attempted independently after setup.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import time

# Workers must use this notebook's interpreter, and every timestamp here is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()

from pyspark.sql import SparkSession, Window, functions as F

spark = (
    SparkSession.builder.appName("pyspark-challenges")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # tiny data needs few partitions
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
MAX_TS = F.lit("9999-12-31 00:00:00").cast("timestamp")
print("Spark", spark.version)

## Challenge 1 — Clean and transform customer records

Trim and lowercase email addresses, replace a null or blank email with `unknown`, and assign `adult` for age 18 or greater, `minor` below 18, and `unknown` for a null age. Return `id`, `email_clean`, and `age_group`, one row per customer.

### Sample data

In [ ]:
customers = spark.createDataFrame([(1, '  AVA@EXAMPLE.COM ', 22), (2, '  ', 16), (3, None, None)], 'id int, email string, age int')
customers.show(truncate=False)

### Expected output

| id | email_clean | age_group |
|---:|---|---|
| 1 | ava@example.com | adult |
| 2 | unknown | minor |
| 3 | unknown | unknown |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
result = customers.select(
    'id',
    F.when(F.trim(F.col('email')) == '', 'unknown')
     .otherwise(F.coalesce(F.lower(F.trim('email')), F.lit('unknown'))).alias('email_clean'),
    F.when(F.col('age').isNull(), 'unknown').when(F.col('age') >= 18, 'adult')
     .otherwise('minor').alias('age_group'))
assert [tuple(r) for r in result.orderBy('id').collect()] == [
    (1, 'ava@example.com', 'adult'), (2, 'unknown', 'minor'), (3, 'unknown', 'unknown')]
result.orderBy('id').show(truncate=False)
```

</details>

## Challenge 2 — Nulls: row counts and averages

For each team, report the number of rows, the number of non-null scores, and the average of the non-null scores. Return null for an all-null average; do not replace missing scores with zero.

### Sample data

In [ ]:
scores = spark.createDataFrame([('A', 10), ('A', None), ('A', 20), ('B', None)], 'team string, score int')
scores.show()

### Expected output

| team | rows | known_scores | avg_score |
|---|---:|---:|---:|
| A | 3 | 2 | 15.0 |
| B | 1 | 0 | null |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
result = scores.groupBy('team').agg(F.count('*').alias('rows'), F.count('score').alias('known_scores'), F.avg('score').alias('avg_score'))
assert [tuple(r) for r in result.orderBy('team').collect()] == [('A', 3, 2, 15.0), ('B', 1, 0, None)]
result.orderBy('team').show()
```

</details>

## Challenge 3 — Parse dates and calculate elapsed days

Parse `placed` and `shipped` as dates, then calculate `days_to_ship`. Preserve null shipment dates and return `order_id`, `placed_date`, and `days_to_ship`.

### Sample data

In [ ]:
shipments = spark.createDataFrame([('o1', '2026-01-30', '2026-02-02'), ('o2', '2026-02-28', None), ('o3', '2026-03-01', '2026-03-01')], ['order_id', 'placed', 'shipped'])
shipments.show()

### Expected output

| order_id | placed_date | days_to_ship |
|---|---|---:|
| o1 | 2026-01-30 | 3 |
| o2 | 2026-02-28 | null |
| o3 | 2026-03-01 | 0 |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
result = shipments.select('order_id', F.to_date('placed').alias('placed_date'), F.datediff(F.to_date('shipped'), F.to_date('placed')).alias('days_to_ship'))
assert [(r.order_id, r.placed_date.isoformat(), r.days_to_ship) for r in result.orderBy('order_id').collect()] == [('o1', '2026-01-30', 3), ('o2', '2026-02-28', None), ('o3', '2026-03-01', 0)]
result.orderBy('order_id').show()
```

</details>

## Challenge 4 — Left joins and missing matches

List every customer with the number of orders. Return zero for customers with no orders, and do not count the null placeholder produced by a left join.

### Sample data

In [ ]:
join_customers = spark.createDataFrame([(1, 'Ava'), (2, 'Bo'), (3, 'Cy')], ['customer_id', 'name'])
join_orders = spark.createDataFrame([(101, 1), (102, 1), (103, 2)], ['order_id', 'customer_id'])
join_customers.show(); join_orders.show()

### Expected output

| customer_id | name | order_count |
|---:|---|---:|
| 1 | Ava | 2 |
| 2 | Bo | 1 |
| 3 | Cy | 0 |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
result = (join_customers.join(join_orders, 'customer_id', 'left')
          .groupBy('customer_id', 'name').agg(F.count('order_id').alias('order_count')))
assert [tuple(r) for r in result.orderBy('customer_id').collect()] == [(1, 'Ava', 2), (2, 'Bo', 1), (3, 'Cy', 0)]
result.orderBy('customer_id').show()
```

</details>

## Challenge 5 — Anti join: customers without purchases

Return the customer IDs with no matching order using a left anti join. A null key in the order table must not exclude all customers.

### Sample data

In [ ]:
anti_customers = spark.createDataFrame([(1,), (2,), (3,)], 'customer_id int')
anti_orders = spark.createDataFrame([(1,), (None,)], 'customer_id int')
anti_customers.show(); anti_orders.show()

### Expected output

| customer_id |
|---:|
| 2 |
| 3 |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
result = anti_customers.join(anti_orders, 'customer_id', 'left_anti')
assert [r.customer_id for r in result.orderBy('customer_id').collect()] == [2, 3]
result.orderBy('customer_id').show()
```

</details>

## Challenge 6 — Spark SQL: CTEs and conditional aggregation

Register the orders as a temporary view. Use a CTE to select paid orders, then return total paid revenue per region and retain only regions with revenue at least 100. The view is session-scoped; no persistent table is required.

### Sample data

In [ ]:
sales = spark.createDataFrame([('east', 'paid', 80), ('east', 'paid', 40), ('west', 'paid', 90), ('west', 'cancelled', 30)], ['region', 'status', 'amount'])
sales.show()

### Expected output

| region | revenue |
|---|---:|
| east | 120 |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
sales.createOrReplaceTempView('challenge_sales')
result = spark.sql("""WITH paid AS (SELECT region, amount FROM challenge_sales WHERE status = 'paid')
SELECT region, SUM(amount) AS revenue FROM paid GROUP BY region HAVING SUM(amount) >= 100""")
assert [tuple(r) for r in result.collect()] == [('east', 120)]
result.show()
```

</details>

## Challenge 7 — Window functions: top two per department

Return the top two employees per department by salary. Break ties by employee ID ascending so that exactly two rows are selected where available. Return department, employee_id, salary, and rank.

### Sample data

In [ ]:
employees = spark.createDataFrame([('A', 1, 100), ('A', 2, 100), ('A', 3, 80), ('B', 4, 90)], ['department', 'employee_id', 'salary'])
employees.show()

### Expected output

| department | employee_id | salary | rank |
|---|---:|---:|---:|
| A | 1 | 100 | 1 |
| A | 2 | 100 | 2 |
| B | 4 | 90 | 1 |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
w = Window.partitionBy('department').orderBy(F.desc('salary'), F.asc('employee_id'))
result = employees.withColumn('rank', F.row_number().over(w)).filter(F.col('rank') <= 2)
assert [tuple(r) for r in result.orderBy('department', 'rank').collect()] == [('A', 1, 100, 1), ('A', 2, 100, 2), ('B', 4, 90, 1)]
result.orderBy('department', 'rank').show()
```

</details>

## Challenge 8 — Running totals with an explicit window frame

Compute a running amount per account ordered by transaction date and ID. Treat null amounts as zero. Use a ROWS frame so each transaction advances the total individually.

### Sample data

In [ ]:
transactions = spark.createDataFrame([('a', 1, '2026-01-01', 5), ('a', 2, '2026-01-01', None), ('a', 3, '2026-01-02', 7), ('b', 4, '2026-01-01', 2)], 'account string, tx_id int, tx_date string, amount int')
transactions.show()

### Expected output

| account | tx_id | running_total |
|---|---:|---:|
| a | 1 | 5 |
| a | 2 | 5 |
| a | 3 | 12 |
| b | 4 | 2 |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
w = Window.partitionBy('account').orderBy('tx_date', 'tx_id').rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = transactions.select('account', 'tx_id', F.sum(F.coalesce('amount', F.lit(0))).over(w).alias('running_total'))
assert [tuple(r) for r in result.orderBy('account', 'tx_id').collect()] == [('a', 1, 5), ('a', 2, 5), ('a', 3, 12), ('b', 4, 2)]
result.orderBy('account', 'tx_id').show()
```

</details>

## Challenge 9 — Arrays and explode_outer

Return one row per customer and tag. Preserve customers with an empty or null array as one row with a null tag. Output `customer_id`, `tag`.

### Sample data

In [ ]:
tagged = spark.createDataFrame([(1, ['sql', 'spark']), (2, []), (3, None)], 'customer_id int, tags array<string>')
tagged.show(truncate=False)

### Expected output

| customer_id | tag |
|---:|---|
| 1 | spark |
| 1 | sql |
| 2 | null |
| 3 | null |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
result = tagged.select('customer_id', F.explode_outer('tags').alias('tag'))
assert [tuple(r) for r in result.orderBy('customer_id', 'tag').collect()] == [(1, 'spark'), (1, 'sql'), (2, None), (3, None)]
result.orderBy('customer_id', 'tag').show()
```

</details>

## Challenge 10 — Union evolving schemas by name

Combine the two feeds by column name. The old feed has no `source` column: represent that value as null. Return `id`, `amount`, `source`.

### Sample data

In [ ]:
old_feed = spark.createDataFrame([(1, 10)], ['id', 'amount'])
new_feed = spark.createDataFrame([('web', 2, 20)], ['source', 'id', 'amount'])
old_feed.show(); new_feed.show()

### Expected output

| id | amount | source |
|---:|---:|---|
| 1 | 10 | null |
| 2 | 20 | web |

### Your solution

Write and run your solution below.

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

One possible implementation and a check:

```python
result = old_feed.unionByName(new_feed, allowMissingColumns=True).select('id', 'amount', 'source')
assert [tuple(r) for r in result.orderBy('id').collect()] == [(1, 10, None), (2, 20, 'web')]
result.orderBy('id').show()
```

</details>

## Challenge 11 - Latest customer record with deterministic deduplication

The customer feed contains retries and out-of-order updates. Return exactly one row per `customer_id`, choosing
the latest `updated_at`. If timestamps tie, choose the greatest `ingest_id`. Keep customers whose email is null.

Requirements:

- Do not use `dropDuplicates` without defining which record wins.
- Say how the result changes if two records share `customer_id`, `updated_at` and `ingest_id`. Then the winner is
  arbitrary, so the pipeline should fail a check rather than pick one silently.
- Add a data-quality check that detects ties on the complete winning ordering key before choosing a winner.

**Expected grain:** one row per customer.

In [ ]:
customer_updates = spark.createDataFrame([
    ('c1', 'ava@old.example', '2026-10-01 09:00:00', 10),
    ('c1', 'ava@example.com', '2026-10-01 09:00:00', 11),
    ('c2', None, '2026-10-02 10:00:00', 20),
    ('c2', 'ben@example.com', '2026-10-01 10:00:00', 19),
    ('c3', 'cy@example.com', '2026-10-03 12:00:00', 30)
], ['customer_id', 'email', 'updated_at', 'ingest_id'])
customer_updates.show(truncate=False)

### Expected output

| customer_id | email | updated_at | ingest_id |
|---|---|---|---:|
| c1 | ava@example.com | 2026-10-01 09:00:00 | 11 |
| c2 | null | 2026-10-02 10:00:00 | 20 |
| c3 | cy@example.com | 2026-10-03 12:00:00 | 30 |

### Your solution

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

Reference solution. Window on the key, order by the tie-break chain, keep rank 1.

```python
# Check for ties before row_number hides them.
winning_keys = (customer_updates.withColumn('updated_ts', F.to_timestamp('updated_at'))
    .withColumn('winner_rank', F.dense_rank().over(Window.partitionBy('customer_id').orderBy(F.desc('updated_ts'), F.desc('ingest_id'))))
    .filter(F.col('winner_rank') == 1))
assert winning_keys.groupBy('customer_id').count().filter(F.col('count') > 1).count() == 0

latest_customers = (
    customer_updates
    .withColumn("updated_ts", F.to_timestamp("updated_at"))
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id")
            .orderBy(F.col("updated_ts").desc(), F.col("ingest_id").desc())
        ),
    )
    .filter(F.col("rn") == 1)
    .drop("rn", "updated_ts")
)
latest_customers.orderBy("customer_id").show(truncate=False)

rows = {r["customer_id"]: r for r in latest_customers.collect()}
assert latest_customers.count() == 3
assert rows["c1"]["ingest_id"] == 11 and rows["c1"]["email"] == "ava@example.com"
assert rows["c2"]["email"] is None                 # null email is kept, not dropped
assert rows["c3"]["ingest_id"] == 30

# Data-quality gate: exactly one winning record per key.
duplicate_winners = latest_customers.groupBy("customer_id").count().filter(F.col("count") > 1)
assert duplicate_winners.count() == 0
```

</details>

## Challenge 12 - Orders joined to a many-to-many category dimension

Produce one row per order with `order_total`, `discount_total` and `net_total`. Each order has line items, and a
product can carry several category labels. The labels feed a separate category summary, but they must not
multiply order revenue.

Requirements:

- Compute line revenue at the line grain before any join to categories.
- Keep orders whose product has no category.
- Build the category summary separately, one row per category. A line in two categories counts in both, so
  category totals are not additive. Say so in the answer.
- Prove that adding the category relationship leaves the sum of order totals unchanged.

**Expected grains:** one row per order for the order summary; one row per category for the category summary.

In [ ]:
order_lines = spark.createDataFrame([
    ('o1', 'p1', 2, 10.0), ('o1', 'p2', 1, 25.0),
    ('o2', 'p1', 1, 10.0), ('o3', 'p3', 1, 8.0)
], ['order_id', 'product_id', 'quantity', 'unit_price'])
discounts = spark.createDataFrame([
    ('o1', 'd1', 5.0), ('o1', 'd2', 2.0), ('o3', 'd3', 1.0)
], ['order_id', 'discount_id', 'discount_amount'])
product_categories = spark.createDataFrame([
    ('p1', 'home'), ('p1', 'sale'), ('p2', 'home')
], ['product_id', 'category'])
order_lines.show(); discounts.show(); product_categories.show()

### Expected output

**Orders:** `(o1, 45, 7, 38)`, `(o2, 10, 0, 10)`, `(o3, 8, 1, 7)` for `(order_id, order_total, discount_total, net_total)`. **Category revenue:** `(home, 55)`, `(sale, 30)`; categories overlap and are not additive.

### Your solution

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

Reference solution. Revenue is computed per line, discounts are pre-aggregated to one row per order, and only then
are the joins made. That keeps the join from fanning out the order rows.

```python
lines = order_lines.withColumn("line_revenue", F.col("quantity") * F.col("unit_price"))

discount_by_order = discounts.groupBy("order_id").agg(F.sum("discount_amount").alias("discount_total"))

order_summary = (
    lines.groupBy("order_id").agg(F.sum("line_revenue").alias("order_total"))
    .join(discount_by_order, "order_id", "left")
    .withColumn("discount_total", F.coalesce("discount_total", F.lit(0.0)))
    .withColumn("net_total", F.col("order_total") - F.col("discount_total"))
    .orderBy("order_id")
)
order_summary.show()

category_summary = (
    lines.join(product_categories, "product_id", "inner")
    .groupBy("category").agg(F.sum("line_revenue").alias("category_revenue"))
    .orderBy("category")
)
category_summary.show()

by_order = {r["order_id"]: r for r in order_summary.collect()}
assert order_summary.count() == 3                  # o3 is kept although its product has no category
assert by_order["o1"]["order_total"] == 45.0 and by_order["o1"]["net_total"] == 38.0
assert by_order["o2"]["discount_total"] == 0.0
assert by_order["o3"]["net_total"] == 7.0

by_category = {r["category"]: r["category_revenue"] for r in category_summary.collect()}
assert by_category == {"home": 55.0, "sale": 30.0}

# Conservation: the category join does not change order revenue.
order_revenue = order_summary.agg(F.sum("order_total")).first()[0]
line_revenue = lines.agg(F.sum("line_revenue")).first()[0]
assert order_revenue == line_revenue == 63.0
```

</details>

## Challenge 13 - Slowly changing dimension as-of join

Attach the customer segment that was valid when each event occurred. Dimension versions use the half-open interval
`[effective_from, effective_to)`, and a null `effective_to` means the version is current.

Requirements:

- Keep events that have no matching version.
- An event exactly at `effective_to` belongs to the next version.
- Detect overlapping versions of the same customer before the join, because overlaps would fan out the result.
- Return exactly one row per event.

**Expected grain:** one row per event.

In [ ]:
events = spark.createDataFrame([
    ('e1', 'c1', '2026-01-15 12:00:00', 20.0),
    ('e2', 'c1', '2026-07-01 12:00:00', 30.0),
    ('e3', 'c2', '2026-03-01 12:00:00', 40.0),
    ('e4', 'missing', '2026-03-01 12:00:00', 50.0)
], ['event_id', 'customer_id', 'event_ts', 'amount'])
segment_history = spark.createDataFrame([
    ('c1', 'standard', '2026-01-01 00:00:00', '2026-06-01 00:00:00'),
    ('c1', 'premium', '2026-06-01 00:00:00', None),
    ('c2', 'standard', '2026-01-01 00:00:00', None)
], ['customer_id', 'segment', 'effective_from', 'effective_to'])
events.show(); segment_history.show()

### Expected output

| event_id | segment |
|---|---|
| e1 | standard |
| e2 | premium |
| e3 | standard |
| e4 | null |

### Your solution

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

Reference solution. Two versions overlap when, for the earlier one `a` and the later one `b` of the same customer,
`b.effective_from` falls before `a.effective_to`. Check that first. Then join on the half-open interval.

```python
# Duplicate start times are ambiguous and must also fail the quality gate.
assert segment_history.groupBy('customer_id', 'effective_from').count().filter(F.col('count') > 1).count() == 0

history = (segment_history
    .withColumn("effective_from", F.to_timestamp("effective_from"))
    .withColumn("effective_to", F.to_timestamp("effective_to")))
events_ts = events.withColumn("event_ts", F.to_timestamp("event_ts"))

a = history.alias("a")
b = history.alias("b")
overlapping = a.join(
    b,
    (F.col("a.customer_id") == F.col("b.customer_id"))
    & (F.col("a.effective_from") < F.col("b.effective_from"))
    & (F.col("b.effective_from") < F.coalesce(F.col("a.effective_to"), MAX_TS)),
)
assert overlapping.count() == 0, "overlapping segment versions would fan out the join"

in_version = (
    (F.col("e.customer_id") == F.col("h.customer_id"))
    & (F.col("e.event_ts") >= F.col("h.effective_from"))
    & (F.col("e.event_ts") < F.coalesce(F.col("h.effective_to"), MAX_TS))
)
priced = (events_ts.alias("e")
    .join(history.alias("h"), in_version, "left")
    .select("e.event_id", "e.customer_id", "e.event_ts", "e.amount", "h.segment"))
priced.orderBy("event_id").show(truncate=False)

segments = {r["event_id"]: r["segment"] for r in priced.collect()}
assert priced.count() == 4                         # one row per event, no fan-out
assert segments == {"e1": "standard", "e2": "premium", "e3": "standard", "e4": None}

# Boundary: an event exactly at effective_to belongs to the next version.
boundary = (spark.createDataFrame([("eb", "c1", "2026-06-01 00:00:00", 1.0)],
                                  ["event_id", "customer_id", "event_ts", "amount"])
    .withColumn("event_ts", F.to_timestamp("event_ts")))
boundary_row = (boundary.alias("e")
    .join(history.alias("h"), in_version, "left")
    .select("h.segment").first())
assert boundary_row["segment"] == "premium"
```

</details>

## Challenge 14 - Sessionize events without leaking across users

Create sessions from web events. A new session starts when the gap from the previous event of the same user is
greater than 30 minutes (exactly 30 minutes stays in the session). Return one row per session with its start,
end, event count and total value.

Requirements:

- Window by user and order by timestamp, then event id, so the result is deterministic.
- Events of one user must never affect another user's sessions.
- A null value counts as zero in the session total.
- Explain the cost of the window and how you would scale it when one user has billions of events.

**Expected grain:** one row per user session.

In [ ]:
web_events = spark.createDataFrame([
    ('u1', 'a', '2026-10-01 10:00:00', 4.0),
    ('u1', 'b', '2026-10-01 10:20:00', None),
    ('u1', 'c', '2026-10-01 11:00:01', 7.0),
    ('u2', 'd', '2026-10-01 10:05:00', 3.0),
    ('u2', 'e', '2026-10-01 10:35:00', 5.0)
], ['user_id', 'event_id', 'event_ts', 'value'])
web_events.show()

### Expected output

| user_id | session_no | start | end | count | total |
|---|---:|---|---|---:|---:|
| u1 | 1 | 10:00:00 | 10:20:00 | 2 | 4 |
| u1 | 2 | 11:00:01 | 11:00:01 | 1 | 7 |
| u2 | 1 | 10:05:00 | 10:35:00 | 2 | 8 |

All timestamps are on 2026-10-01 UTC.

### Your solution

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

Reference solution. `lag` finds the previous event, a flag marks each session start, and a running sum of the flag
numbers the sessions. The window is partitioned by user, so no session can cross users. The running sum is a
window over the user's full history: that is the cost to explain, because one very large user is one partition.

```python
ordered = Window.partitionBy("user_id").orderBy("event_ts", "event_id")
events_ts = web_events.withColumn("event_ts", F.to_timestamp("event_ts"))

with_gap = events_ts.withColumn("prev_ts", F.lag("event_ts").over(ordered))
is_new_session = F.when(
    F.col("prev_ts").isNull()
    | (F.unix_timestamp("event_ts") - F.unix_timestamp("prev_ts") > 30 * 60),
    1,
).otherwise(0)

sessions = (
    with_gap.withColumn("is_new", is_new_session)
    .withColumn("session_no", F.sum("is_new").over(ordered.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .groupBy("user_id", "session_no")
    .agg(
        F.min("event_ts").alias("session_start"),
        F.max("event_ts").alias("session_end"),
        F.count("*").alias("event_count"),
        F.sum(F.coalesce("value", F.lit(0.0))).alias("session_value"),
    )
    .orderBy("user_id", "session_no")
)
sessions.show(truncate=False)

rows = [tuple(r) for r in sessions.select("user_id", "event_count", "session_value").collect()]
assert rows == [("u1", 2, 4.0), ("u1", 1, 7.0), ("u2", 2, 8.0)]
```

</details>

## Challenge 15 - Skewed join with selective salting

A fact table is joined to a dimension on `account_id`. Profiling shows that `UNKNOWN` holds most fact rows, so one
task would do most of the join work. Implement a salted join that gives the same result as the unsalted join.

Requirements:

- Salt only the hot key, with a deterministic bucket from `event_id`.
- Replicate only the hot dimension row across the salt range, not the whole dimension.
- Compare row counts and totals with a baseline unsalted inner join.
- Say when broadcasting the dimension or enabling AQE skew join would be better.
- Say how to choose the salt count from Spark UI metrics (task durations and shuffle read sizes), not by guessing.

**Expected grain:** one row per matching fact row.

In [ ]:
fact_events = spark.createDataFrame([
    ('e1', 'UNKNOWN', 10), ('e2', 'UNKNOWN', 20), ('e3', 'UNKNOWN', 30),
    ('e4', 'a1', 5), ('e5', 'a2', 7), ('e6', None, 11)
], ['event_id', 'account_id', 'metric'])
accounts = spark.createDataFrame([
    ('UNKNOWN', 'fallback'), ('a1', 'alpha'), ('a2', 'beta')
], ['account_id', 'account_type'])
fact_events.show(); accounts.show()

### Expected output

The salted inner join returns events `e1`–`e5`: 5 rows and metric total 72. The null-key event `e6` does not match.

### Your solution

In [ ]:
# Write your solution here

<details>
<summary>Show reference solution</summary>

Reference solution. Hot-key fact rows get a salt from `pmod(hash(event_id), N)`. The hot dimension row is
cross-joined with the salt range, and cold keys keep salt 0. The join then matches on `(account_id, salt)`.

```python
SALT_BUCKETS = 3
HOT_KEY = "UNKNOWN"

baseline = fact_events.join(accounts, "account_id", "inner")

fact_salted = fact_events.withColumn(
    "salt",
    F.when(F.col("account_id") == HOT_KEY, F.pmod(F.hash("event_id"), F.lit(SALT_BUCKETS)))
    .otherwise(F.lit(0)),
)
salts = spark.range(SALT_BUCKETS).select(F.col("id").cast("int").alias("salt"))
hot_dimension = accounts.filter(F.col("account_id") == HOT_KEY).crossJoin(salts)
cold_dimension = accounts.filter(F.col("account_id") != HOT_KEY).withColumn("salt", F.lit(0))
dimension_salted = hot_dimension.unionByName(cold_dimension)

salted = (
    fact_salted.join(dimension_salted, ["account_id", "salt"], "inner")
    .select("event_id", "account_id", "metric", "account_type")
)

# Baseline equality: same rows, same count, same total.
assert salted.count() == baseline.count() == 5     # the null-account row is dropped by both
assert salted.agg(F.sum("metric")).first()[0] == baseline.agg(F.sum("metric")).first()[0] == 72
assert {r.event_id for r in salted.collect()} == {r.event_id for r in baseline.collect()}
```

</details>

## Interviewer notes

Keep the reference solutions and adversarial fixtures outside the candidate copy. Change values, column names,
duplicate patterns and null placement, but keep the schema stable. Judge the answer on grain, duplicate resistance,
null semantics, conservation of totals, deterministic tie-breaking, the physical plan (shuffles, broadcast, skew)
and testing, not only on whether the sample output looks right. Ask the candidate to rerun after changing the salt
count, or after adding a duplicate dimension record.